**Part A - Load and Inspect the data set**

1. Load the Titanic dataset into a pandas DataFrame.

In [24]:
import pandas as pd 
import numpy as np


from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix


In [25]:

#given 
URLS = ["titanic.csv", "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"]

df = None

for url in URLS:
    try:
        df = pd.read_csv(url)
        print("data loaded from", url)
        break
    except Exception as e:
        print("Failed to load data from", url, "error:", e)

if df is None:
    raise RuntimeError("Failed to load data")



data loaded from titanic.csv


2. Show:
- first 5 rows (head())
- shape (rows x columns)
- column names
- data types (dtypes)

In [26]:
#show first 5 rows of the dataframe
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [27]:
#show shape of the dataframe
print("Shape:", df.shape)

#show column names
print("Columns:", list(df.columns))

#show data types
print("Datatypes:", df.dtypes)

Shape: (891, 12)
Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
Datatypes: PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object


3. Identify which columns are:
 - numeric
    - PassengerId, Age, SibSp, Parch, Fare
 - categorical
    - Pclass, Survived, Name, Sex,  Ticket, Cabin, Embarked
 - contain missing values

**Required output:** a short table showing **missing value counts per coumn**

In [30]:
nan = df.isna().sum().sort_values(ascending=False)
nan[nan > 0]


Cabin       687
Age         177
Embarked      2
dtype: int64

**Part B - Handle missing values**

Handle missing values using the following rules:

1. **Age:** fill missing values with the **median age.**
2. **Embarked:** fill missing values with the **most frequent value (mode).**
3. **Drop columns:** remove **Cabin** (too many missing values)

    **Required output**:
    - missing value counts **before** and **after**
    - a brief note (2-3 sentences) explaining what you did (no theory required)

In [38]:
# create a copy of our dataframe for data integrity
X = df.copy()

# missing value count before
print("Missing values before processing \n")
nan = X.isna().sum()
print(nan[nan > 0])

# Fill missing Age values with median Age
X['Age'] = X['Age'].fillna(X['Age'].median())

# Fill missing Embarked values with the mode
mode = X['Embarked'].mode()
fillMode = mode.iloc[0] if not mode.empty else "Unknown"
X['Embarked'] = X['Embarked'].fillna(fillMode)

# Drop Cabin column
X = X.drop('Cabin',axis=1)


print("Missing values after processing \n")
nan = X.isna().sum()
print(nan[nan > 0])





Missing values before processing 

Age         177
Cabin       687
Embarked      2
dtype: int64
Missing values after processing 

Series([], dtype: int64)


in the above code we first print out a table of missing values to show the current state of NaN/Nill values within our dataframe.
we then fill the Age column na values with the median using the fillna and .median() functions to retrieve and fill the data, following that we attempt to get the mode of the embarked column, which returns a series - we then retrieve the first index of that series with .iloc[0] to retrieve our fill value - perform a basic check to ensure it has a value before filling all missing values in that column with the mode. .drop() function allows us to drop a row or column from a dataframe - we select the desired column here (Cabin) and pass axis=1 to drop the column before finally attempting to print all null values left in the table

**Part C - Encode categorical variables**
1. Select these categorical columns:
- Sex
- Embarked
2. Apply **one-hot encoding** (using pandas get_dummies()).

    **Required output:**
    - list of final feature column names after encoding
    - show the first 5 rows of the encoded dataset

In [40]:
#create sublist of our selected columns
selected_columns = ['Sex','Embarked']

#one-hot encode our copied X dataframe passing in the selected columns for the column parameter
x_encoded = pd.get_dummies(X,columns=selected_columns)

#print list of final feature columns
print(x_encoded.columns)

#show first 5 rows of encoded dataset
x_encoded.head()

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Sex_female', 'Sex_male', 'Embarked_C', 'Embarked_Q',
       'Embarked_S'],
      dtype='str')


,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,1,0,3,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.2500,False,True,False,False,True
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,0,PC 17599,71.2833,True,False,True,False,False
2,3,1,3,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.9250,True,False,False,False,True
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1000,True,False,False,False,True
4,5,0,3,"Allen, Mr. William Henry",35.0,0,0,373450,8.0500,False,True,False,False,True


**Part D - Scale numeric variables**
Scale these numeric columns using **StandardScaler:**
- Age
- Fare

**Required output:**
- show summary statistics(mean/std) for Age and Fare **before scaling**
- show summary statistics **after scaling** (they don't need to be perfect 0/1 - just show the transformation happened)
    

**Part E - Train/test split**
    Split the dataset into **80% train / 20% test** using: 

- random_state = 42    

Use:
- x = all processed feature columns
    
- y = Survived

**Required output:**
Print shapes of:
- X_train, X_test, y_train,  y_test